# AI Resume Screening System — From Scratch

**Main development notebook**

This notebook demonstrates a small, explainable resume-to-job-description matching system. The algorithms are implemented in the accompanying `resume_screening.py` file using Python's standard library. No pretrained model, online dataset, third-party NLP toolkit, or external AI API is used.

**Important:** This is an educational decision-support prototype. Its scores are not probabilities of job performance and must not be used to automatically accept or reject applicants.

## Learning sequence
1. Load the job description and fictional sample resumes.
2. Extract job requirements and candidate information.
3. Tokenize text and implement TF-IDF.
4. Calculate cosine similarity.
5. Combine transparent component scores and rank candidates.
6. Export the results and test the code.

Run the notebook from the project folder so that `resume_screening.py` and the `data/` folder are visible.


## 1. Import our own implementation

In [ ]:
from pathlib import Path
from pprint import pprint

from resume_screening import (
    read_text_file,
    load_resume_folder,
    remove_personal_information_lines,
    tokenize,
    find_skills,
    extract_education,
    extract_experience_years,
    parse_job_requirements,
    build_tfidf_vectors,
    cosine_similarity,
    calculate_text_similarities,
    screen_resumes,
    print_results,
    export_results_csv,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "resume_screening.py").exists():
    raise FileNotFoundError(
        "Open this notebook with the project folder as the working directory."
    )

JOB_DESCRIPTION_PATH = PROJECT_ROOT / "data" / "job_description.txt"
RESUME_FOLDER = PROJECT_ROOT / "data" / "resumes"
OUTPUT_FOLDER = PROJECT_ROOT / "outputs"
OUTPUT_FOLDER.mkdir(exist_ok=True)

print("Project folder:", PROJECT_ROOT)
print("Python standard-library implementation loaded.")


## 2. Load the job description and resumes

The starter job description and four resumes are synthetic examples written specifically for this project. They do not contain real applicant records.

In [ ]:
job_description = read_text_file(JOB_DESCRIPTION_PATH)
resumes = load_resume_folder(RESUME_FOLDER)

print("JOB DESCRIPTION")
print("-" * 60)
print(job_description)

print("\nRESUME FILES")
for candidate_id, resume_text in resumes.items():
    print(f"{candidate_id}: {len(resume_text)} characters")


## 3. Extract job requirements

We keep the skill dictionary small and visible in `resume_screening.py`. A line under a `Required Skills` heading is treated as required; a line under a `Preferred Skills` heading is treated as preferred. Skills without a priority label default to required. Always inspect the extracted result because this is a rule-based parser.

In [ ]:
requirements = parse_job_requirements(job_description)
pprint(requirements)


## 4. Inspect candidate information

The extractor looks for known skill aliases, simple education terms, and explicitly stated durations such as `2 years of professional experience`. It does not calculate experience from job dates.

In [ ]:
for candidate_id, resume_text in resumes.items():
    print("\n", candidate_id)
    print("Skills:", sorted(find_skills(resume_text)))
    print("Education:")
    pprint(extract_education(resume_text))
    print("Explicit experience years:", extract_experience_years(resume_text))


## 5. Text preprocessing and tokenization

Tokenization lowercases text, removes common filler words, and keeps simple alphanumeric/technical tokens. Labelled personal-information lines and common contact identifiers are removed before similarity is calculated. The skill, education, and experience functions also ignore those labelled lines.

In [ ]:
example_resume = next(iter(resumes.values()))
print("Original excerpt:")
print(example_resume[:350])
print("\nTokens used for text similarity:")
print(tokenize(example_resume)[:60])


## 6. Build TF-IDF vectors ourselves

For each term `t` in document `d`:

- `TF(t,d) = count(t in d) / number of tokens in d`
- `IDF(t) = log((N + 1) / (df(t) + 1)) + 1`
- `TF-IDF(t,d) = TF(t,d) * IDF(t)`

Each document vector is a Python dictionary mapping a term to its TF-IDF weight. This implementation does not call a ready-made vectorizer.

In [ ]:
documents = [job_description] + list(resumes.values())
tfidf_vectors = build_tfidf_vectors(documents)

print("Number of document vectors:", len(tfidf_vectors))
print("Number of distinct terms in the job-description vector:", len(tfidf_vectors[0]))
print("A few job-description TF-IDF values:")
for term, value in list(tfidf_vectors[0].items())[:15]:
    print(f"{term:20s} {value:.4f}")


## 7. Calculate cosine similarity

Cosine similarity compares the direction of two term-weight vectors. It is 0 when there are no shared terms and closer to 1 when the word-weight patterns are more similar. It is not a measure of candidate quality by itself.

In [ ]:
job_vector = tfidf_vectors[0]
print("Text similarities (0 to 1):")
for (candidate_id, _), vector in zip(resumes.items(), tfidf_vectors[1:]):
    similarity = cosine_similarity(job_vector, vector)
    print(f"{candidate_id:20s} {similarity:.4f}")


## 8. Calculate scores and rank candidates

The base weights are 60% skills, 20% TF-IDF text similarity, 12% experience, and 8% education. If the job description does not state an experience or education criterion, that score component is omitted and the remaining weights are normalized. Within the skill component, required coverage is 75% and preferred coverage is 25% when both groups exist.

In [ ]:
report = screen_resumes(job_description, resumes)
print_results(report)


## 9. Inspect one candidate's explanation

Always look beyond the final percentage. The matched/missing skills and source evidence help a human reviewer understand why candidates differ.

In [ ]:
for candidate in report["candidates"]:
    print("\n" + "=" * 70)
    print(f"Rank {candidate['rank']}: {candidate['candidate_id']}")
    print("Overall score:", candidate["overall_score"], "%")
    print("Component scores:")
    pprint(candidate["components"])
    print("Matched required skills:", candidate["required_skills_matched"])
    print("Missing required skills:", candidate["required_skills_missing"])
    print("Matched preferred skills:", candidate["preferred_skills_matched"])
    print("Education:")
    pprint(candidate["education"])
    print("Experience evidence:", candidate["experience_evidence"])


## 10. Export a CSV report

The CSV contains scores and explanation fields. Keep the file in a secure location if you later test with real resumes.

In [ ]:
csv_path = export_results_csv(
    report, OUTPUT_FOLDER / "candidate_screening_results.csv"
)
print("CSV report written to:", csv_path)


## 11. Try your own input

For this first version, create one plain-text file per resume (`candidate_005.txt`, etc.) in `data/resumes/` and edit `data/job_description.txt`. Use neutral candidate IDs rather than names. Then rerun the notebook from the beginning.

You can also experiment directly in this cell by replacing the strings below. Do not paste real personal data into a shared notebook or cloud runtime without authorization.

In [ ]:
# Example template — replace these fictional strings with your own test text.
custom_job_description = """
Required Skills: Python, SQL, Microsoft Excel
Preferred Skills: Power BI, Pandas
At least 1 year of professional experience is required.
Bachelor's degree in Computer Science or Statistics preferred.
"""

custom_resumes = {
    "candidate_demo_001": """
    Skills: Python, SQL, Excel, Pandas
    Experience: 2 years of professional experience.
    Education: Bachelor of Science in Statistics.
    """,
    "candidate_demo_002": """
    Skills: Excel, Power BI
    Experience: 1 year of professional experience.
    Education: Diploma in Business Administration.
    """,
}

custom_report = screen_resumes(custom_job_description, custom_resumes)
print_results(custom_report)


## 12. Run automated tests

The separate `tests/` folder checks the key functions. Run the command below in a terminal opened at the project root:

```bash
python -m unittest discover -s tests -v
```

The initial project version includes 11 tests. Add a test whenever you change an algorithm or add a new rule.

## 13. Limitations and responsible use

- The sample dataset is synthetic and small; it is only for demonstrating the program.
- The skill dictionary and text rules must be checked against each job description.
- Text similarity measures shared words, not deep meaning.
- The education matcher is a simple keyword rule and cannot establish credential equivalency.
- Experience is recognized only when explicitly stated.
- The initial system accepts `.txt` files; it does not parse PDF/DOCX or perform OCR.
- A score is not a probability of employee performance. It must not be used to automatically reject applicants.
- Do not use age, gender, race, religion, caste, disability, marital status, nationality, photos, or other protected/sensitive traits. Have a human review the original evidence and apply consistent, lawful criteria.

## 14. Where the implementation lives

All algorithms are in `resume_screening.py`: `tokenize`, `find_skills`, `extract_experience_years`, `extract_education`, `build_tfidf_vectors`, `cosine_similarity`, `calculate_skill_score`, `screen_resumes`, and the export functions. Read that file alongside this notebook to explain the implementation line by line.